# CSIRO Biomass image inference

Generate `submission.csv` from a locally trained model uploaded to a Kaggle Dataset.

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import timm
from tqdm.auto import tqdm
from PIL import Image, ImageOps
# ML系
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
import gc

warnings.filterwarnings('ignore')

# ==========================================
# 0. 共通設定
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

COMP_DIR = "/kaggle/input/csiro-biomass-prediction"
if not os.path.exists(os.path.join(COMP_DIR, "test.csv")):
    COMP_DIR = "/kaggle/input/csiro-biomass"

# v4特徴量
DATASET_DIR = "/kaggle/input/koro2jp"
if not os.path.exists(DATASET_DIR):
    DATASET_DIR = "/kaggle/input/csiro-biomass-models"

# modelv3
MODEL_V3_DIR = "/kaggle/input/modelv3"

TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

# ======================================================================================
# PART 1: Model V3 (FiLM DINOv3 Expert) - Deep Learning
# ======================================================================================
print("\n" + "="*40)
print("PART 1: Running Model V3 (FiLM DINOv3 Expert)")
print("="*40)

class FiLM(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2), nn.ReLU(inplace=True), nn.Linear(feat_dim // 2, feat_dim * 2)
        )
    def forward(self, context):
        return torch.chunk(self.mlp(context), 2, dim=1)

class CSIROModelRegressor(nn.Module):
    def __init__(self, model_name, pretrained=False, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.film = FiLM(self.backbone.num_features)
        # Single Head for Expert Model
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features * 2, 8), nn.ReLU(inplace=True), 
            nn.Dropout(0.0), nn.Linear(8, 1)
        )
        self.softplus = nn.Softplus(beta=1.0)

    def forward(self, left_img, right_img):
        l, r = self.backbone(left_img), self.backbone(right_img)
        gamma, beta = self.film((l + r) / 2)
        # Single value output
        return self.softplus(self.head(torch.cat([l * (1 + gamma) + beta, r * (1 + gamma) + beta], dim=1)))

class RegressionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df, self.img_dir, self.transform = df, img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.df.iloc[idx]['image_path']))
        try: img = Image.open(path).convert('RGB')
        except: img = Image.new("RGB", (2000, 1000))
        w, h = img.size
        l, r = img.crop((0, 0, w//2, h)), img.crop((w//2, 0, w, h))
        if self.transform: l, r = self.transform(l), self.transform(r)
        return l, r

def run_model_v3():
    test_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
    unique_test = test_df[['image_path']].drop_duplicates().reset_index(drop=True)
    
    trans = T.Compose([T.Resize((512, 512)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    loader = DataLoader(RegressionDataset(unique_test, os.path.join(COMP_DIR, "test"), trans), batch_size=32, num_workers=2)
    
    preds_v3 = np.zeros((len(unique_test), 5)) # [Clover, Dead, Green, Total, GDM]
    
    # Init Model (Single Head)
    model = CSIROModelRegressor("vit_large_patch16_dinov3_qkvb", pretrained=False, num_classes=1).to(DEVICE)
    
    # Target Mapping: Output index -> Target Name
    # We need predictions for: Clover(0), Dead(1), Green(2)
    targets = ['clover', 'dead', 'green']
    target_indices = [0, 1, 2] 
    
    for t_name, t_idx in zip(targets, target_indices):
        print(f"  Predicting {t_name.upper()}...")
        fold_preds = []
        # Find weights for this target across folds
        weights = glob.glob(f"{MODEL_V3_DIR}/**/fold*_{t_name}.pth", recursive=True)
        
        if not weights:
            print(f"  ⚠️ No weights found for {t_name}")
            continue
            
        for w_path in weights:
            # Load Weight
            state = torch.load(w_path, map_location=DEVICE)
            model.load_state_dict(state)
            model.eval()
            
            p = []
            with torch.no_grad():
                for l, r in loader:
                    out = model(l.to(DEVICE), r.to(DEVICE))
                    p.append(out.cpu().numpy().flatten())
            fold_preds.append(np.concatenate(p))
            
        if fold_preds:
            # Average across folds
            preds_v3[:, t_idx] = np.mean(fold_preds, axis=0)
            
    # Mass Balance: Calc GDM & Total
    preds_v3[:, 4] = preds_v3[:, 2] + preds_v3[:, 0] # GDM = Green + Clover
    preds_v3[:, 3] = preds_v3[:, 4] + preds_v3[:, 1] # Total = GDM + Dead
    
    return preds_v3

preds_v3 = run_model_v3()
print("✅ Model V3 Done.")
gc.collect()
torch.cuda.empty_cache()

# ======================================================================================
# PART 2: 0.64 Base (SigLIP + DINOv2 + CLIP) - NO ConvNeXt
# ======================================================================================
print("\n" + "="*40)
print("PART 2: Running 0.64 Base (v4 Features Only)")
print("="*40)

def extract_features_base():
    # Helper to load HF models and extract
    def get_hf_emb(model_name):
        try:
            if "clip" in model_name:
                proc = CLIPProcessor.from_pretrained(model_name)
                mod = CLIPModel.from_pretrained(model_name).to(DEVICE).eval()
            else:
                proc = AutoImageProcessor.from_pretrained(model_name)
                mod = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
            
            emb = []
            test_paths = [os.path.join(COMP_DIR, p) for p in pd.read_csv(os.path.join(COMP_DIR, "test.csv"))[['image_path']].drop_duplicates()['image_path']]
            
            for i in tqdm(range(0, len(test_paths), 32), desc=f"Ext {model_name[:10]}"):
                batch = test_paths[i:i+32]
                imgs = []
                for p in batch:
                    try: imgs.append(Image.open(p).convert("RGB"))
                    except: imgs.append(Image.new("RGB", (224, 224)))
                
                with torch.no_grad():
                    if "clip" in model_name:
                        out = mod.get_image_features(**proc(images=imgs, return_tensors="pt", padding=True).to(DEVICE))
                    elif "siglip" in model_name:
                        out = mod.get_image_features(**proc(images=imgs, return_tensors="pt").to(DEVICE))
                    else: # DINOv2
                        o = mod(**proc(images=imgs, return_tensors="pt").to(DEVICE))
                        out = torch.cat([o.last_hidden_state[:,0], o.last_hidden_state[:,1:].mean(1)], 1)
                    emb.append((out / out.norm(dim=-1, keepdim=True)).cpu().numpy())
            return np.vstack(emb)
        except: return None

    def find(k): 
        f = glob.glob(f"/kaggle/input/**/*{k}*", recursive=True)
        return f[0] if f else k
    
    # Only use trusted 3 models
    e1 = get_hf_emb(find("siglip"))
    e2 = get_hf_emb(find("dinov2"))
    e3 = get_hf_emb(find("clip"))
    
    embs = [e for e in [e1, e2, e3] if e is not None]
    if not embs: return None
    return np.hstack(embs).astype(np.float32)

def run_model_base():
    X_test = extract_features_base()
    if X_test is None: return preds_v3 # Fallback
    
    # Load Train Features (v4 Only)
    print("  Loading Train Features (v4)...")
    try:
        v4_path = glob.glob(f"{DATASET_DIR}/**/train_embeddings_large_v4.csv", recursive=True)[0]
        df_train_feats = pd.read_csv(v4_path)
        
        # ConvNeXt is NOT loaded here
        
        df_tr = pd.read_csv(os.path.join(COMP_DIR, "train.csv"))
        if 'target_name' in df_tr.columns:
            df_tr = df_tr.pivot_table(index='image_path', columns='target_name', values='target').reset_index()
        
        df_train = pd.merge(df_train_feats, df_tr, on='image_path')
        
        feat_cols = [c for c in df_train.columns if c.startswith('emb')]
        X_train = df_train[feat_cols].values.astype(np.float32)
        y_train = df_train[TARGET_NAMES].values.astype(np.float32)
        
        if X_test.shape[1] != X_train.shape[1]:
            dim = min(X_test.shape[1], X_train.shape[1])
            X_test, X_train = X_test[:, :dim], X_train[:, :dim]
            
    except: return np.zeros((len(X_test), 5))

    # Training (Ridge + GBDT)
    print("  Training Ensemble...")
    y_log = np.log1p(y_train)
    
    # Norm & PCA (Safe settings from 0.64)
    norm = Normalizer()
    X_tr_n = norm.fit_transform(X_train)
    X_te_n = norm.transform(X_test)
    
    pca = PCA(n_components=50, random_state=42) # 50 dim is safe for v4
    X_tr_p = pca.fit_transform(X_tr_n)
    X_te_p = pca.transform(X_te_n)
    
    preds = np.zeros((len(X_test), 5))
    
    # Ridge
    r = MultiOutputRegressor(RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0])).fit(X_tr_p, y_log)
    preds += r.predict(X_te_p) * 0.4
    
    # GBDTs
    for i in range(5):
        l = lgb.LGBMRegressor(n_estimators=500, verbose=-1, random_state=42).fit(X_tr_p, y_log[:, i])
        preds[:, i] += l.predict(X_te_p) * 0.2
        
        x = xgb.XGBRegressor(n_estimators=500, verbosity=0, random_state=42, device=DEVICE).fit(X_tr_p, y_log[:, i])
        preds[:, i] += x.predict(X_te_p) * 0.2
        
        try: c = CatBoostRegressor(iterations=500, verbose=0, random_state=42, task_type='GPU').fit(X_tr_p, y_log[:, i])
        except: c = CatBoostRegressor(iterations=500, verbose=0, random_state=42, task_type='CPU').fit(X_tr_p, y_log[:, i])
        preds[:, i] += c.predict(X_te_p) * 0.2
        
    return np.maximum(0, np.expm1(preds))

preds_base = run_model_base()
print("✅ 0.64 Base (No ConvNeXt) Done.")

# ======================================================================================
# PART 3: Final Ensemble
# ======================================================================================
print("\n" + "="*40)
print("PART 3: Blending & Saving")
print("="*40)

# 重み付け: Model V3 (Deep Learning) を主軸に、Base (ML) で安定させる
W_V3 = 0.65
W_BASE = 0.35

final_preds = preds_v3 * W_V3 + preds_base * W_BASE

# Mass Balance Check
# GDM = Green + Clover
final_preds[:, 4] = final_preds[:, 2] + final_preds[:, 0]
# Total = GDM + Dead
final_preds[:, 3] = final_preds[:, 4] + final_preds[:, 1]

# Save
sub_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
vals = []
unique_test = sub_df[['image_path']].drop_duplicates().reset_index(drop=True)

for i, row in sub_df.iterrows():
    idx = unique_test[unique_test['image_path'] == row['image_path']].index[0]
    t_idx = TARGET_NAMES.index(row['target_name'])
    vals.append(final_preds[idx, t_idx])

sub_df['target'] = vals

# Safety Clipping
REAL_MAX = {'Dry_Clover_g': 72, 'Dry_Dead_g': 84, 'Dry_Green_g': 158, 'Dry_Total_g': 186, 'GDM_g': 158}
for t, m in REAL_MAX.items():
    sub_df.loc[sub_df['target_name']==t, 'target'] = sub_df.loc[sub_df['target_name']==t, 'target'].clip(upper=m*1.2)

sub_df[['sample_id', 'target']].to_csv("submission.csv", index=False)
print("✅ Final Submission Created!")
print(sub_df.head())
